# Análisis de grafos de llamadas y detección de SCC

Este cuaderno de trabajo explora las funciones avanzadas de análisis de código de UnifyWeaver:

- **Construcción de grafos de llamadas** - Creación de grafos de dependencias a partir de código Prolog
- **Detección de SCC** - Búsqueda de componentes fuertemente conexas (recursión mutua)
- **Análisis de patrones** - Comprensión de patrones de recursión
- **Visualización de dependencias** - Visualización de relaciones entre predicados

## Objetivos de aprendizaje

- Comprender cómo analiza UnifyWeaver la estructura del código
- Construir e inspeccionar grafos de llamadas
- Detectar recursión mutua utilizando el algoritmo de Tarjan
- Visualizar dependencias de código

## Configuración

Carga UnifyWeaver y los módulos de análisis.

In [ ]:
% Cargar inicialización
['../init'].

% Cargar los módulos de análisis
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Ejemplo 1: Grafo de llamadas simple

Comencemos con un predicado simple y construyamos su grafo de llamadas.

In [ ]:
% Definir el predicado ancestor
:- dynamic ancestor/2.
:- dynamic parent/2.

% Hechos de parent
parent(abraham, isaac).
parent(isaac, jacob).

% Reglas de ancestor
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### Construir el grafo de llamadas

In [ ]:
% Construir grafo de llamadas para ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Analizar dependencias

In [ ]:
% Obtener todas las dependencias de ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Comprobar si es autorrecursivo
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## Ejemplo 2: Detección de recursión mutua

Ahora detectemos la recursión mutua con el ejemplo de par/impar.

In [ ]:
% Definir predicados mutuamente recursivos
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### Construir el grafo de llamadas para ambos predicados

In [ ]:
% Construir grafo de llamadas para ambos predicados
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Encontrar componentes fuertemente conexas (SCC)

In [ ]:
% Reconstruir el grafo porque las variables no persisten entre celdas del cuaderno
build_call_graph([is_even/1, is_odd/1], _Graph),
% Encontrar SCCs usando el algoritmo de Tarjan
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### Comprobar si la SCC es trivial

In [ ]:
% Reconstruir los valores derivados para que esta celda también se ejecute de forma independiente
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Comprobar cada SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## Ejemplo 3: Grafo de llamadas complejo

Analicemos un sistema más complejo con múltiples predicados.

In [ ]:
% Definir un programa pequeño con múltiples predicados
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent usa parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: mismo progenitor, diferentes hijos
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: los progenitores son hermanos
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### Construir el grafo de llamadas completo

In [ ]:
% Construir grafo de llamadas para todos los predicados
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Encontrar grupos de predicados

Encuentra el grupo de predicados mutuamente recursivos que contiene un predicado inicial.

In [ ]:
% Encontrar el grupo mutuamente recursivo que contiene cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## Ejemplo 4: Detección de patrones

Usemos comparadores de patrones para analizar tipos de recursión.

In [ ]:
% Definir varios patrones recursivos
:- dynamic count/3.     % Recursivo de cola
:- dynamic factorial/2. % Recursivo lineal
:- dynamic fib/2.       % Recursivo de árbol (o lineal si se detecta)

% Conteo con recursión de cola
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Factorial con recursión lineal
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (puede detectarse como lineal o de árbol)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### Detectar recursión de cola

In [ ]:
% Comprobar si count/3 es recursivo de cola
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### Detectar recursión lineal

In [ ]:
% Comprobar si factorial/2 es recursivo lineal
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### Contar llamadas recursivas

In [ ]:
% Contar llamadas recursivas en fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## Visualización con formato DOT

Generemos una representación Graphviz DOT de nuestro grafo de llamadas.

In [ ]:
% Función auxiliar para generar formato DOT
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Generar DOT para el grafo par/impar
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### Guardar el archivo DOT

In [ ]:
% Reconstruir el código fuente de DOT porque las variables no persisten entre celdas
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## Ejercicio: ¡Analiza tu propio código!

¡Prueba definir tus propios predicados y analizarlos!

In [ ]:
% Define tus predicados aquí
% Luego construye grafos de llamadas, encuentra SCCs y detecta patrones

% Ejemplo:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## Resumen

En este cuaderno de trabajo, aprendiste:

✅ Cómo construir grafos de llamadas a partir de código Prolog

✅ Cómo detectar componentes fuertemente conexas (SCC) para recursión mutua

✅ Cómo usar comparadores de patrones para clasificar tipos de recursión

✅ Cómo analizar dependencias entre predicados

✅ Cómo visualizar grafos de llamadas con el formato DOT

## Temas avanzados

Para análisis más avanzados:

- **Ordenación topológica**: Usa `topological_order/2` para ordenar las SCC por dependencias
- **Comparadores de patrones personalizados**: Escribe tus propios predicados de detección de patrones
- **Extracción de patrones de acumulador**: Usa `extract_accumulator_pattern/2` para análisis detallado
- **Prohibir recursión lineal**: Usa `forbid_linear_recursion/1` para forzar diferentes estrategias de compilación

## Referencias

- Capítulo 10: Introspección y teoría de Prolog
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`